In [1]:
%pwd

'c:\\PROJECTS\\End_to_end_Kidney_Disease_Classification_Deep_learning_project\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\PROJECTS\\End_to_end_Kidney_Disease_Classification_Deep_learning_project'

In [24]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class TrainingConfig:
    root_dir:Path
    trained_model_path:Path
    updated_base_model_path: Path
    training_data:Path
    params_epochs: int
    params_batch_size:int
    params_is_augmentation:bool
    params_image_size:list
    params_learning_rate: float 

In [25]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories
import tensorflow as tf

In [26]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:

        config = self.config
        params = self.params

        training_data = os.path.join(
            config.data_ingestion.unzip_dir,
            "DATASET_KIDNEY"
        )

        create_directories([
            Path(config.training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(config.training.root_dir),
            trained_model_path=Path(config.training.trained_model_path),
            updated_base_model_path=Path(config.prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),

            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
            params_learning_rate=params.LEARNING_RATE   # 🔥 IMPORTANT FIX
        )

        return training_config

In [27]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [28]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

        # compile model properly
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=self.config.params_learning_rate
            ),
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        # validation generator
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            class_mode="categorical",   # 🔥 IMPORTANT FIX
            **dataflow_kwargs
        )

        # training generator
        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            class_mode="categorical",   # 🔥 IMPORTANT FIX
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):

        # 🔴 SAFETY FIX (avoid zero steps)
        self.steps_per_epoch = max(
            1,
            self.train_generator.samples // self.train_generator.batch_size
        )

        self.validation_steps = max(
            1,
            self.valid_generator.samples // self.valid_generator.batch_size
        )

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [30]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training=Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    print("TRAINING COMPLETED")
except Exception as e:
        raise e

[2026-05-16 19:01:09,519: INFO: common: Yaml file loaded successfully]
[2026-05-16 19:01:09,531: INFO: common: Yaml file loaded successfully]
[2026-05-16 19:01:09,536: INFO: common: Created Directory:artifacts]
[2026-05-16 19:01:09,543: INFO: common: Created Directory:artifacts\training]
[2026-05-16 19:01:10,102: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.
368/368 ━━━━━━━━━━━━━━━━━━━━ 1649s 4s/step - accuracy: 0.7935 - loss: 3.2640 - val_accuracy: 0.6971 - val_loss: 13.0113
[2026-05-16 19:28:43,559: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model,